# A historical rental market in classifieds
**The Rental Price Puzzle · Market evidence**  
Cyril James Lunzaga

This notebook reads the reproducible aggregate study tables. It does not expose individual rental listings. Figures and numerical results describe US classified advertisements from 2018–2019, not current or executed rental prices.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "portfolio" / "tables").is_dir():
    project_root = project_root.parent
tables = project_root / "portfolio" / "tables"
assert tables.is_dir(), "Open from the repository root or notebooks folder, and generate portfolio tables first"
flow = pd.read_csv(tables / "cohort_flow.csv")
quantiles = pd.read_csv(tables / "rent_quantiles.csv")
missing = pd.read_csv(tables / "field_missingness.csv")
display(flow)
display(quantiles)

## Population and distribution

The verified cohort starts with 99,492 records. The 99,403 distinct eligible listings remain an exploratory population, while 99,101 with state and city information support the geographic modeling design. Exact duplicate removal is not equivalent to identifying unique apartments.

In [ ]:
price = quantiles.loc[quantiles["measurement"].eq("asking_rent_usd_month")].iloc[0]
print(f"Median asking rent: ${price['p50']:,.0f}/month")
print(f"Middle 50%: ${price['p25']:,.0f}–${price['p75']:,.0f}/month")
print(f"99th percentile / maximum: ${price['p99']:,.0f} / ${price['p100']:,.0f}")
assert price["p100"] >= price["p99"] >= price["p50"]

In [ ]:
top_missing = missing.loc[missing["missing_count"].gt(0)].head(8).sort_values("missing_percent")
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top_missing["field"], top_missing["missing_percent"])
ax.set(xlabel="Missing rows (%)", title="Incomplete listing attributes")
fig.tight_layout()
plt.show()

## Geographic differences are descriptive

Locations are not sampled evenly. Grouped medians therefore describe the collection of observed advertisements, not a weighted national rental index. The charts below do not adjust for size, amenities, or data-source composition.

In [ ]:
states = pd.read_csv(tables / "state_rent_summary.csv")
representative = states.head(12).sort_values("listings")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(representative["state"], representative["listings"])
ax.set(xlabel="Listings in audited location cohort", title="Most represented state codes")
fig.tight_layout()
plt.show()
display(states.head(10))

**Interpretation.** Rent and floor-area upper tails are substantial, and several listing details are missing. The primary model retains high-value observations; the separate diagnostic tables describe errors on those extreme listings without replacing the full-sample evaluation.